# Modulo 4 · Lezione 4 — Progettazione di basi di dati

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vedremo oggi?

| # | Argomento | Cosa imparerai |
|---|------|----------------|
| 1 | Processo di progettazione | Concettuale → Logico → Fisico |
| 2 | Tipi di relazioni | 1:1, 1:N, N:M |
| 3 | Prima Forma Normale (1FN) | Senza gruppi ripetitivi |
| 4 | Seconda Forma Normale (2FN) | Senza dipendenze parziali |
| 5 | Terza Forma Normale (3FN) | Senza dipendenze transitive |
| 6 | Quando denormalizzare | Trade-off nella pratica |
| 7 | Pratica | Progettare il DB di un sistema universitario |

---
## 1. Il processo di progettazione

```
┌─────────────────────────────────────────────────────────────┐
│ 1. ANALISI DEI REQUISITI                                     │
│    «Quali informazioni dobbiamo memorizzare e a che scopo?» │
└───────────────────────────┬─────────────────────────────────┘
                            ▼
┌─────────────────────────────────────────────────────────────┐
│ 2. MODELLO CONCETTUALE — Diagramma ER                        │
│    Entità, attributi e relazioni (ancora senza SQL)          │
└───────────────────────────┬─────────────────────────────────┘
                            ▼
┌─────────────────────────────────────────────────────────────┐
│ 3. MODELLO LOGICO — Tabelle normalizzate                     │
│    Definire tabelle, colonne, tipi, chiavi, vincoli          │
└───────────────────────────┬─────────────────────────────────┘
                            ▼
┌─────────────────────────────────────────────────────────────┐
│ 4. MODELLO FISICO — DDL specifico del motore                 │
│    CREATE TABLE, indici, partizioni, ecc.                    │
└─────────────────────────────────────────────────────────────┘
```

In questa lezione ci concentriamo sui modelli **concettuale** e **logico**: come pensare la progettazione prima di scrivere SQL.

---
## 2. Tipi di relazioni

### Relazione 1:1 (uno a uno)

Ogni riga di A corrisponde al massimo a una riga di B, e viceversa.

```
empleados ──── 1:1 ──── fichas_medicas

Implementazione: FK nella tabella con più campi opzionali, con UNIQUE

empleados(id, nombre, ...)
fichas_medicas(id, empleado_id UNIQUE, grupo_sanguineo, ...)
```

### Relazione 1:N (uno a molti) — la più comune

Una riga di A può avere molte righe di B collegate, ma ogni riga di B appartiene a una sola di A.

```
clientes ──── 1:N ──── pedidos

Un cliente → molti ordini
Un ordine  → un cliente

Implementazione: FK nella tabella sul lato «N»
pedidos(id, cliente_id, ...)   ← FK in pedidos
```

### Relazione N:M (molti a molti)

Una riga di A può essere collegata a molte di B, e una di B a molte di A.

```
estudiantes ──── N:M ──── asignaturas

Uno studente → molte materie
Una materia → molti studenti

Implementazione: TABELLA INTERMEDIA (tabella ponte)

estudiantes(id, nombre)
asignaturas(id, nombre)
matriculas(estudiante_id, asignatura_id, fecha, nota)  ← tabella ponte
         PK composta: (estudiante_id, asignatura_id)
```

---
## 3. Normalizzazione

La normalizzazione è il processo di organizzare le tabelle per ridurre la **ridondanza** e migliorare l'**integrità** dei dati.

Lavoreremo con questa tabella denormalizzata come esempio:

```
TABELLA: pedidos_raw (tabella piatta, tutto in una)
┌────────┬───────────┬─────────────────┬─────────────┬────────────────────┬─────────┬──────────┬──────┐
│ped_id  │cli_nombre │cli_email        │ cli_ciudad  │ productos           │cantidad │precio    │fecha │
├────────┼───────────┼─────────────────┼─────────────┼────────────────────┼─────────┼──────────┼──────┤
│ 1      │ Ana       │ ana@mail.com    │ Madrid      │ Teclado, Ratón      │ 1,1     │ 85,45    │01/01 │
│ 2      │ Ana       │ ana@mail.com    │ Madrid      │ Smartphone          │ 1       │ 599      │15/01 │
│ 3      │ Carlos    │ carlos@mail.com │ Barcelona   │ Monitor, Teclado    │ 1,2     │ 450,85   │20/01 │
└────────┴───────────┴─────────────────┴─────────────┴────────────────────┴─────────┴──────────┴──────┘

Problemi:
❌ «Ana» e la sua email compaiono in più righe (ridondanza)
❌ «productos» e «cantidad» contengono liste separate da virgole (gruppi ripetitivi)
❌ Se Ana cambia email, bisogna aggiornare più righe
❌ Impossibile interrogare «ordini che includono Teclado» con SQL
```

---
## 3. Prima Forma Normale (1FN)

**Regole della 1FN:**
1. Ogni cella contiene un **unico valore atomico** (niente liste, niente insiemi).
2. Ogni colonna contiene valori dello **stesso tipo**.
3. Ogni riga è **unica** (c'è una chiave primaria).

```
PRIMA della 1FN (violazione: più prodotti in una cella):
┌───────┬──────────┬─────────────────────┬─────────┐
│ped_id │ cliente  │ productos            │cantidad │
├───────┼──────────┼─────────────────────┼─────────┤
│  1    │ Ana      │ Teclado, Ratón       │ 1, 1    │

DOPO la 1FN (una riga per prodotto):
┌───────┬──────────┬───────────┬─────────┐
│ped_id │ cliente  │ producto  │cantidad │
├───────┼──────────┼───────────┼─────────┤
│  1    │ Ana      │ Teclado   │  1      │
│  1    │ Ana      │ Ratón     │  1      │
└───────┴──────────┴───────────┴─────────┘

PK: (ped_id, producto)  — chiave primaria composta
```

---
## 4. Seconda Forma Normale (2FN)

Requisito preliminare: essere in 1FN.

**Regola della 2FN:** Tutti gli attributi non chiave devono dipendere da **tutta** la chiave primaria, non da una sua parte.

Si applica solo quando la PK è composta.

```
Tabella in 1FN con PK = (ped_id, producto_id):
┌────────┬────────────┬──────────┬───────────┬──────────────┬──────────┐
│ped_id  │producto_id │ cantidad │ cli_nombre│ prod_nombre  │ precio   │
├────────┼────────────┼──────────┼───────────┼──────────────┼──────────┤
│  1     │    5       │   1      │ Ana       │ Teclado      │  85.00   │

Dipendenze:
  cantidad    → dipende da (ped_id, producto_id) ✅ — dipende da TUTTA la PK
  cli_nombre  → dipende solo da ped_id ❌ — dipendenza PARZIALE
  prod_nombre → dipende solo da producto_id ❌ — dipendenza PARZIALE
  precio      → dipende solo da producto_id ❌ — dipendenza PARZIALE

SOLUZIONE: separare in tre tabelle
  pedidos(ped_id, cliente_id, fecha)
  productos(producto_id, nombre, precio)
  lineas_pedido(ped_id, producto_id, cantidad)   ← solo ciò che dipende da TUTTA la PK
```

---
## 5. Terza Forma Normale (3FN)

Requisito preliminare: essere in 2FN.

**Regola della 3FN:** Non devono esserci dipendenze transitive. Un attributo non chiave non deve dipendere da un altro attributo non chiave.

```
Tabella in 2FN:
clientes(id, nombre, ciudad, codigo_postal, pais)

Dipendenze:
  nombre, ciudad → dipendono da id ✅
  pais           → dipende da codigo_postal ❌  (dipendenza TRANSITIVA)
                   id → codigo_postal → pais

SOLUZIONE:
  clientes(id, nombre, ciudad, codigo_postal)
  codigos_postales(codigo_postal, pais, provincia)  ← estrarre la dipendenza transitiva
```

**Regola pratica:** Se il dato X determina il dato Y, e nessuno dei due è la PK, estrai una nuova tabella.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

conn = sqlite3.connect(":memory:")
cur  = conn.cursor()
cur.execute("PRAGMA foreign_keys = ON")

# ── TABELLA DENORMALIZZATA (senza nessuna forma normale) ──────────
cur.execute("""
    CREATE TABLE pedidos_plano (
        pedido_id     INTEGER,
        cli_nombre    TEXT,
        cli_email     TEXT,
        cli_ciudad    TEXT,
        prod_nombres  TEXT,     -- ❌ lista: "Teclado, Ratón"
        cantidades    TEXT,     -- ❌ lista: "1, 1"
        precios       TEXT,     -- ❌ lista: "85, 45"
        fecha         TEXT
    )
""")

cur.executemany("INSERT INTO pedidos_plano VALUES (?,?,?,?,?,?,?,?)", [
    (1, 'Ana', 'ana@mail.com', 'Madrid',    'Teclado, Ratón',   '1, 1', '85, 45', '2025-01-10'),
    (2, 'Ana', 'ana@mail.com', 'Madrid',    'Smartphone',       '1',    '599',    '2025-01-15'),
    (3, 'Carlos', 'c@mail.com','Barcelona', 'Monitor, Teclado', '1, 2', '450, 85','2025-01-20'),
])

print("Tabella denormalizzata:")
display(pd.read_sql_query("SELECT * FROM pedidos_plano", conn))
print("\nProblemi: liste nelle celle, dati del cliente ripetuti.")

In [ ]:
# ── SCHEMA NORMALIZZATO (3FN) ──────────────────────────────────
cur.executescript("""
    CREATE TABLE clientes (
        id     INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        email  TEXT NOT NULL UNIQUE,
        ciudad TEXT
    );
    CREATE TABLE productos (
        id     INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        precio REAL NOT NULL
    );
    CREATE TABLE pedidos (
        id         INTEGER PRIMARY KEY,
        cliente_id INTEGER NOT NULL REFERENCES clientes(id),
        fecha      TEXT
    );
    CREATE TABLE lineas_pedido (
        pedido_id   INTEGER NOT NULL REFERENCES pedidos(id),
        producto_id INTEGER NOT NULL REFERENCES productos(id),
        cantidad    INTEGER NOT NULL,
        precio_ud   REAL    NOT NULL,
        PRIMARY KEY (pedido_id, producto_id)
    );
""")

# Popolamento
cur.executemany("INSERT INTO clientes VALUES (?,?,?,?)", [
    (1, 'Ana',    'ana@mail.com', 'Madrid'),
    (2, 'Carlos', 'c@mail.com',   'Barcelona'),
])
cur.executemany("INSERT INTO productos VALUES (?,?,?)", [
    (1, 'Teclado',    85.00),
    (2, 'Ratón',      45.00),
    (3, 'Smartphone', 599.00),
    (4, 'Monitor',    450.00),
])
cur.executemany("INSERT INTO pedidos VALUES (?,?,?)", [
    (1, 1, '2025-01-10'),
    (2, 1, '2025-01-15'),
    (3, 2, '2025-01-20'),
])
cur.executemany("INSERT INTO lineas_pedido VALUES (?,?,?,?)", [
    (1, 1, 1, 85.00),
    (1, 2, 1, 45.00),
    (2, 3, 1, 599.00),
    (3, 4, 1, 450.00),
    (3, 1, 2, 85.00),
])
conn.commit()

# Ora la stessa informazione è perfettamente interrogabile
print("Schema normalizzato — dettaglio ordini:")
display(pd.read_sql_query("""
    SELECT p.id AS pedido, c.nombre AS cliente, prod.nombre AS producto,
           lp.cantidad, lp.precio_ud,
           ROUND(lp.cantidad * lp.precio_ud, 2) AS subtotal
    FROM lineas_pedido lp
    JOIN pedidos   p    ON lp.pedido_id   = p.id
    JOIN clientes  c    ON p.cliente_id   = c.id
    JOIN productos prod ON lp.producto_id = prod.id
    ORDER BY p.id, prod.nombre
""", conn))

conn.close()

---
## 6. Quando denormalizzare

La normalizzazione è la regola generale, ma ha delle eccezioni:

| Situazione | Può giustificare la denormalizzazione |
|-----------|-------------------------------|
| Query di lettura molto frequenti | Ridurre JOIN costosi |
| Data warehouse / analitica | Le tabelle piatte sono più veloci per le aggregazioni |
| Colonne che cambiano raramente | La ridondanza ha un basso costo di manutenzione |
| Auditing / storico | Salvare il prezzo al momento della vendita |

```
Esempio di denormalizzazione giustificata:

lineas_pedido.precio_ud  ← salviamo il prezzo al momento della vendita
                           (anche se è già presente in productos.precio)

Perché? Se domani cambia il prezzo del prodotto, l'importo
dell'ordine storico deve riflettere il prezzo pagato dal cliente, non quello attuale.
```

> Regola pratica: **normalizza prima, denormalizza solo quando le prestazioni lo richiedono** e puoi misurarne il beneficio.

---
## Diagramma ER — esempio completo

```
┌─────────────┐        ┌─────────────────┐        ┌─────────────┐
│  clientes   │        │    pedidos      │        │  productos  │
├─────────────┤        ├─────────────────┤        ├─────────────┤
│ PK id       │──1──┐  │ PK id           │  ┌──N──│ PK id       │
│    nombre   │     └─N│ FK cliente_id   │  │     │    nombre   │
│    email    │        │    fecha        │  │     │    precio   │
│    ciudad   │        │    estado       │  │     │    stock    │
└─────────────┘        └────────┬────────┘  │     │ FK cat_id   │
                                │            │     └──────┬──────┘
                               1│            │N           │N
                                ▼            │            │
                       ┌─────────────────┐  │     ┌──────┴──────┐
                       │ lineas_pedido   │◄─┘     │ categorias  │
                       ├─────────────────┤        ├─────────────┤
                       │ PK FK pedido_id │        │ PK id       │
                       │ PK FK prod_id  │        │    nombre   │
                       │    cantidad    │        └─────────────┘
                       │    precio_ud   │
                       └─────────────────┘

Relazioni:
clientes ──1:N──► pedidos
pedidos ──N:M──► productos  (tramite lineas_pedido)
productos ──N:1──► categorias
```

---
## 7. Pratica — Sistema universitario

---

### Esercizio 1 — Identificare le forme normali

**Consegna:** Analizza questa tabella e stabilisci quali forme normali viola:

```
tabella: inscripciones
┌──────────┬──────────────┬────────────────────────────┬───────┬───────────────┬──────────────────┐
│ alumno_id│ alumno_nombre│ asignaturas                 │notas  │prof_nombre    │ departamento     │
├──────────┼──────────────┼────────────────────────────┼───────┼───────────────┼──────────────────┤
│  1       │ María        │ Matemáticas, Física         │ 8, 7  │ García, López │ Ciencias, Ciencias│
│  2       │ Juan         │ Informática                 │ 9     │ Martínez      │ Tecnología       │
│  3       │ María        │ Informática                 │ 6     │ Martínez      │ Tecnología       │
└──────────┴──────────────┴────────────────────────────┴───────┴───────────────┴──────────────────┘
```

Rispondi in una cella Markdown:
1. Cosa viola la 1FN?
2. Cosa viola la 2FN (assumi PK composta)?
3. Cosa viola la 3FN?
4. Proponi lo schema normalizzato.

✏️ **LA TUA RISPOSTA QUI**

1FN:

2FN:

3FN:

Schema proposto:


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

**1FN — Viola** perché:
- `asignaturas`, `notas`, `prof_nombre`, `departamento` contengono liste di valori in un'unica cella (non sono atomici).

**2FN — Viola** (assumendo PK = (alumno_id, asignatura_id)) perché:
- `alumno_nombre` dipende solo da `alumno_id` → dipendenza parziale.
- `prof_nombre` e `departamento` dipendono solo da `asignatura_id` → dipendenza parziale.

**3FN — Viola** perché:
- `departamento` dipende da `prof_nombre`, non direttamente dalla PK → dipendenza transitiva.

**Schema normalizzato (3FN):**
```
departamentos(id, nombre)
profesores(id, nombre, departamento_id FK)
alumnos(id, nombre, email)
asignaturas(id, nombre, profesor_id FK)
inscripciones(alumno_id FK, asignatura_id FK, nota)  PK(alumno_id, asignatura_id)
```
</details>

---
### Esercizio 2 — Implementare lo schema universitario

**Consegna:** Implementa in SQLite lo schema normalizzato dell'esercizio precedente. Poi inserisci dati di prova:
- Tre dipartimenti, quattro professori, sei studenti, cinque materie.
- Almeno dieci iscrizioni con voti variabili.

In [ ]:
# ✏️ IL TUO CODICE QUI
import sqlite3, pandas as pd
from pathlib import Path


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import sqlite3, pandas as pd
from pathlib import Path

conn = sqlite3.connect(Path.home() / "universidad.db")
cur  = conn.cursor()
cur.execute("PRAGMA foreign_keys = ON")

cur.executescript("""
    DROP TABLE IF EXISTS inscripciones;
    DROP TABLE IF EXISTS asignaturas;
    DROP TABLE IF EXISTS alumnos;
    DROP TABLE IF EXISTS profesores;
    DROP TABLE IF EXISTS departamentos;

    CREATE TABLE departamentos (
        id     INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL UNIQUE
    );
    CREATE TABLE profesores (
        id              INTEGER PRIMARY KEY,
        nombre          TEXT NOT NULL,
        departamento_id INTEGER REFERENCES departamentos(id)
    );
    CREATE TABLE alumnos (
        id     INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        email  TEXT NOT NULL UNIQUE
    );
    CREATE TABLE asignaturas (
        id          INTEGER PRIMARY KEY,
        nombre      TEXT NOT NULL,
        creditos    INTEGER DEFAULT 6,
        profesor_id INTEGER REFERENCES profesores(id)
    );
    CREATE TABLE inscripciones (
        alumno_id     INTEGER NOT NULL REFERENCES alumnos(id),
        asignatura_id INTEGER NOT NULL REFERENCES asignaturas(id),
        nota          REAL CHECK(nota BETWEEN 0 AND 10),
        PRIMARY KEY (alumno_id, asignatura_id)
    );
""")

cur.executemany("INSERT INTO departamentos VALUES (?,?)", [
    (1,'Ciencias'),(2,'Tecnología'),(3,'Humanidades')
])
cur.executemany("INSERT INTO profesores VALUES (?,?,?)", [
    (1,'García',1),(2,'López',1),(3,'Martínez',2),(4,'Ruiz',3)
])
cur.executemany("INSERT INTO alumnos VALUES (?,?,?)", [
    (1,'María','maria@uni.es'),(2,'Juan','juan@uni.es'),
    (3,'Laura','laura@uni.es'),(4,'Pedro','pedro@uni.es'),
    (5,'Sofía','sofia@uni.es'),(6,'Marcos','marcos@uni.es'),
])
cur.executemany("INSERT INTO asignaturas VALUES (?,?,?,?)", [
    (1,'Matemáticas',6,1),(2,'Física',6,2),(3,'Informática',6,3),
    (4,'Literatura',6,4),(5,'Estadística',4,1),
])
cur.executemany("INSERT INTO inscripciones VALUES (?,?,?)", [
    (1,1,8.5),(1,2,7.0),(1,3,6.5),
    (2,3,9.0),(2,5,7.5),
    (3,1,5.0),(3,4,8.0),
    (4,2,4.5),(4,3,7.0),(4,5,6.0),
    (5,1,9.5),(5,3,8.0),
    (6,4,7.5),(6,2,6.0),
])
conn.commit()
print("Database universitario creato")
```
</details>

---
### Esercizio 3 — Query sullo schema

**Consegna:** Con il DB universitario dell'esercizio precedente:
1. Elenca gli studenti con voto medio ≥ 7.
2. Conta quanti studenti sono iscritti a ogni materia.
3. Mostra quali materie insegna ogni dipartimento.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
def q(sql): return pd.read_sql_query(sql, conn)

# 1. Studenti con media >= 7
display(q("""
    SELECT a.nombre, ROUND(AVG(i.nota), 2) AS nota_media
    FROM alumnos a JOIN inscripciones i ON a.id = i.alumno_id
    GROUP BY a.id, a.nombre
    HAVING AVG(i.nota) >= 7
    ORDER BY nota_media DESC
"""))

# 2. Studenti per materia
display(q("""
    SELECT asig.nombre, COUNT(i.alumno_id) AS alumnos
    FROM asignaturas asig
    LEFT JOIN inscripciones i ON asig.id = i.asignatura_id
    GROUP BY asig.id, asig.nombre
    ORDER BY alumnos DESC
"""))

# 3. Materie per dipartimento
display(q("""
    SELECT d.nombre AS departamento, GROUP_CONCAT(asig.nombre, ', ') AS asignaturas
    FROM departamentos d
    JOIN profesores p ON p.departamento_id = d.id
    JOIN asignaturas asig ON asig.profesor_id = p.id
    GROUP BY d.id, d.nombre
"""))
```
</details>

---
### Esercizio 4 — Identificare le relazioni

**Consegna:** Per ogni coppia di entità, identifica il tipo di relazione (1:1, 1:N, N:M) e spiega perché:

1. `paciente` — `historial_medico`
2. `autor` — `libro`
3. `vuelo` — `pasajero`
4. `empleado` — `departamento`
5. `producto` — `etiqueta` (tag)

✏️ **LA TUA RISPOSTA QUI**

1. paciente — historial_medico:
2. autor — libro:
3. vuelo — pasajero:
4. empleado — departamento:
5. producto — etiqueta:


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

1. **paciente — historial_medico: 1:1** — ogni paziente ha un'unica cartella clinica e ogni cartella appartiene a un unico paziente.

2. **autor — libro: 1:N** — un autore può scrivere molti libri, ma ogni libro ha un unico autore principale. (Se assumiamo la coautoria, sarebbe N:M.)

3. **vuelo — pasajero: N:M** — un volo porta molti passeggeri e un passeggero può prendere molti voli. Tabella ponte: `reservas(vuelo_id, pasajero_id, asiento, ...)`.

4. **empleado — departamento: N:1** — molti dipendenti appartengono a un dipartimento, ma ogni dipendente appartiene a uno solo. FK in `empleados.departamento_id`.

5. **producto — etiqueta: N:M** — un prodotto può avere più etichette e la stessa etichetta può essere applicata a più prodotti. Tabella ponte: `producto_etiquetas(producto_id, etiqueta_id)`.
</details>

---
### Esercizio 5 (sfida) — Progettare da zero

**Consegna:** Progetta e implementa la base di dati per una **piattaforma di streaming musicale** con i seguenti requisiti:

- Gli **artisti** possono collaborare su brani (un artista può avere molti brani; un brano può avere più artisti).
- I **brani** appartengono ad **album**. Un album ha un artista principale, ma può includere collaborazioni.
- Gli **utenti** possono creare **playlist** e aggiungere brani.
- Il sistema registra gli **ascolti** (quale utente ha ascoltato quale brano e quando).

Implementa lo schema in SQLite, inserisci dati di prova e scrivi tre query utili (ad es., artisti più ascoltati, brani più popolari, utente con più ascolti).

In [ ]:
# ✏️ IL TUO CODICE QUI
import sqlite3, pandas as pd
from pathlib import Path


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import sqlite3, pandas as pd
from pathlib import Path

conn = sqlite3.connect(Path.home() / "streaming.db")
cur  = conn.cursor()
cur.execute("PRAGMA foreign_keys = ON")

cur.executescript("""
    DROP TABLE IF EXISTS reproducciones;
    DROP TABLE IF EXISTS playlist_canciones;
    DROP TABLE IF EXISTS playlists;
    DROP TABLE IF EXISTS cancion_artistas;
    DROP TABLE IF EXISTS canciones;
    DROP TABLE IF EXISTS albumes;
    DROP TABLE IF EXISTS usuarios;
    DROP TABLE IF EXISTS artistas;

    CREATE TABLE artistas (
        id     INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        pais   TEXT
    );
    CREATE TABLE albumes (
        id           INTEGER PRIMARY KEY,
        titulo       TEXT NOT NULL,
        anio         INTEGER,
        artista_id   INTEGER REFERENCES artistas(id)
    );
    CREATE TABLE canciones (
        id         INTEGER PRIMARY KEY,
        titulo     TEXT NOT NULL,
        duracion_s INTEGER,
        album_id   INTEGER REFERENCES albumes(id)
    );
    CREATE TABLE cancion_artistas (
        cancion_id INTEGER REFERENCES canciones(id),
        artista_id INTEGER REFERENCES artistas(id),
        rol        TEXT DEFAULT 'principal',
        PRIMARY KEY (cancion_id, artista_id)
    );
    CREATE TABLE usuarios (
        id     INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        email  TEXT UNIQUE
    );
    CREATE TABLE playlists (
        id         INTEGER PRIMARY KEY,
        nombre     TEXT NOT NULL,
        usuario_id INTEGER REFERENCES usuarios(id)
    );
    CREATE TABLE playlist_canciones (
        playlist_id INTEGER REFERENCES playlists(id),
        cancion_id  INTEGER REFERENCES canciones(id),
        posicion    INTEGER,
        PRIMARY KEY (playlist_id, cancion_id)
    );
    CREATE TABLE reproducciones (
        id          INTEGER PRIMARY KEY,
        usuario_id  INTEGER REFERENCES usuarios(id),
        cancion_id  INTEGER REFERENCES canciones(id),
        fecha       TEXT DEFAULT (datetime('now'))
    );
""")

# Dati
cur.executemany("INSERT INTO artistas VALUES (?,?,?)", [
    (1,'Taylor Swift','US'),(2,'Bad Bunny','PR'),
    (3,'Rosalía','ES'),(4,'The Weeknd','CA'),
])
cur.executemany("INSERT INTO albumes VALUES (?,?,?,?)", [
    (1,'Midnights',2022,1),(2,'Un Verano Sin Ti',2022,2),
    (3,'El Mal Querer',2018,3),(4,'Dawn FM',2022,4),
])
cur.executemany("INSERT INTO canciones VALUES (?,?,?,?)", [
    (1,'Anti-Hero',197,1),(2,'Lavender Haze',202,1),(3,'Midnight Rain',174,1),
    (4,'Tití Me Preguntó',238,2),(5,'Ojitos Lindos',224,2),
    (6,'Malamente',215,3),(7,'Bagdad',259,3),
    (8,'Sacrifice',189,4),(9,'Less Than Zero',175,4),
])
cur.executemany("INSERT INTO cancion_artistas VALUES (?,?,?)", [
    (1,1,'principal'),(2,1,'principal'),(3,1,'principal'),
    (4,2,'principal'),(5,2,'principal'),(5,3,'colaboración'),  # Ojitos Lindos: Bad Bunny + Rosalía
    (6,3,'principal'),(7,3,'principal'),
    (8,4,'principal'),(9,4,'principal'),
])
cur.executemany("INSERT INTO usuarios VALUES (?,?,?)", [
    (1,'Ana','ana@demo.com'),(2,'Carlos','carlos@demo.com'),(3,'Lucía','lucia@demo.com'),
])
cur.executemany("INSERT INTO playlists VALUES (?,?,?)", [
    (1,'Favoritas',1),(2,'Workout',2),(3,'Relax',3),
])
cur.executemany("INSERT INTO playlist_canciones VALUES (?,?,?)", [
    (1,1,1),(1,6,2),(1,8,3),
    (2,4,1),(2,5,2),(2,8,3),
    (3,2,1),(3,7,2),(3,9,3),
])
# Simuliamo gli ascolti
for uid, cid in [(1,1),(1,1),(1,6),(2,4),(2,4),(2,4),(2,5),(3,8),(3,9),(3,1),(1,8),(2,8)]:
    cur.execute("INSERT INTO reproducciones (usuario_id, cancion_id) VALUES (?,?)", (uid,cid))
conn.commit()

def q(sql): return pd.read_sql_query(sql, conn)

print("=== Top brani ===")
display(q("""
    SELECT c.titulo, COUNT(*) AS reproducciones
    FROM reproducciones r JOIN canciones c ON r.cancion_id = c.id
    GROUP BY c.id, c.titulo ORDER BY reproducciones DESC LIMIT 5
"""))

print("\n=== Artisti più ascoltati ===")
display(q("""
    SELECT a.nombre, COUNT(*) AS reproducciones
    FROM reproducciones r
    JOIN cancion_artistas ca ON r.cancion_id = ca.cancion_id
    JOIN artistas a ON ca.artista_id = a.id
    GROUP BY a.id, a.nombre ORDER BY reproducciones DESC
"""))

print("\n=== Utente più attivo ===")
display(q("""
    SELECT u.nombre, COUNT(*) AS reproducciones
    FROM reproducciones r JOIN usuarios u ON r.usuario_id = u.id
    GROUP BY u.id, u.nombre ORDER BY reproducciones DESC LIMIT 1
"""))
```
</details>

---
## Riepilogo della lezione

```
TIPI DI RELAZIONI
  1:1  → FK + UNIQUE nella tabella secondaria
  1:N  → FK nella tabella sul lato «N»
  N:M  → Tabella ponte con PK composta (fk1, fk2)

FORME NORMALI
  1FN → Valori atomici + PK
  2FN → Senza dipendenze parziali (solo con PK composta)
  3FN → Senza dipendenze transitive tra non-chiave

PROCESSO DI PROGETTAZIONE
  1. Identificare le entità (sostantivi del dominio)
  2. Identificare le relazioni (verbi: «ha», «appartiene a», «acquista»)
  3. Identificare gli attributi di ogni entità
  4. Normalizzare fino alla 3FN
  5. Denormalizzare solo in presenza di un problema di prestazioni comprovato
```

## Cosa vedremo nella prossima lezione?

Nella **Lezione 5** colleghiamo Python ai database SQL: il modulo `sqlite3` della libreria standard e `SQLAlchemy` come ORM.